# Exploración de fuentes de datos

Notebook exploratorio del sistema de monitoreo de calidad del aire.
Carga los 5 CSV de contaminantes y una muestra de la API Open-Meteo.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / "etl").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

from etl.config import get_settings
from etl.extract.csv_extractor import CSV_SOURCES, extract_all_csv
from etl.extract.weather_extractor import extract_weather

settings = get_settings()
sns.set_theme(style="whitegrid")

## 1. Fuentes CSV de contaminantes (5 archivos)

In [ ]:
pollution_dfs = extract_all_csv()
print(f"Fuentes cargadas: {list(pollution_dfs.keys())}")
for pollutant, df in pollution_dfs.items():
    print(f"\n{pollutant}: {len(df)} filas, columnas={list(df.columns)}")
    display(df.head(3))

In [ ]:
profile_rows = []
for pollutant, df in pollution_dfs.items():
    profile_rows.append({
        "pollutant": pollutant,
        "rows": len(df),
        "nulls": int(df.isnull().sum().sum()),
        "fecha_min": df["fecha"].min() if "fecha" in df.columns else None,
        "fecha_max": df["fecha"].max() if "fecha" in df.columns else None,
    })
profile_df = pd.DataFrame(profile_rows)
profile_df

## 2. Fuente API Open-Meteo (clima ERA5)

In [ ]:
weather_df = extract_weather()
print(f"Registros meteorológicos: {len(weather_df)}")
display(weather_df.head())
weather_df.describe()

## 3. Visualización exploratoria

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, (pollutant, df) in zip(axes, pollution_dfs.items()):
    col = "valor_validado" if "valor_validado" in df.columns else df.columns[-1]
    vals = pd.to_numeric(df[col], errors="coerce").dropna()
    ax.hist(vals, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(pollutant)
    ax.set_xlabel("concentración")
if len(pollution_dfs) < len(axes):
    axes[-1].axis("off")
plt.suptitle("Distribución de contaminantes - estación O'Higgins", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
if not weather_df.empty and "temp_max" in weather_df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(weather_df["date"], weather_df["temp_max"], label="temp máx")
    ax.plot(weather_df["date"], weather_df["temp_min"], label="temp mín")
    ax.set_title("Serie temporal de temperatura (Open-Meteo)")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()